# Explore the LNMC MPG TRN Whole-Cell Patch-Clamp Dataset (Comprehensive)

## Introduction
This notebook provides a comprehensive exploratory analysis of the **Single-Cell Dataset of Juvenile Mouse Neurons from the Thalamic Reticular Nucleus — Whole-Cell Patch Electrophysiology**, produced at the EPFL Laboratory of Neural Microcircuitry (LNMC).

Learn more:
- Data Package DOI: [10.71728/senscience.k7f3-9xqm](https://sen.science/doi/10.71728/senscience.k7f3-9xqm/)

As a FAIR² Data Package, it ensures accessibility, interoperability, and AI-readiness. FAIR² datasets follow the MLCommons **Croissant** 🥐 format for machine learning datasets. See the [MLCommons Croissant Format Specification](https://docs.mlcommons.org/croissant/docs/croissant-spec.html).

The dataset provides whole-cell patch-clamp electrophysiology for **22 identified TRN neurons** of juvenile male C57Bl/6J mice, acquired under a standardised current-clamp stimulus battery:
- **22 recording sessions** (one per neuron) with animal identity, age, weight, slice orientation, and resting membrane potential
- **22,918 recorded sweeps** across 54 distinct stimulus protocols
- **527,232,374 digitised signal samples** — the full, un-decimated time series for every sweep

Three tidy Parquet record-sets describe the dataset: `mouse_metadata` (one row per recording session), `sweep_metadata` (one row per sweep), and `sweep_traces` (one row per digitised sample). This notebook loads the metadata tables via the Croissant metadata, links every column back to its method step, and renders a multi-protocol current-clamp trace viewer for a representative cell — reading the raw traces on demand from the `sweep_traces` table.

### Install and import required libraries

In [ ]:
# Install mlcroissant and the plotting/loading dependencies
!pip install mlcroissant pandas pyarrow matplotlib seaborn

In [ ]:
import json
from pathlib import Path

import mlcroissant as mlc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import Markdown, display

sns.set_theme(style="whitegrid")

## 1. Data Loading
We use the `mlcroissant` library to read the FAIR² `fair2.json` metadata, then load the two tidy metadata Parquet record-sets it declares into `pandas` DataFrames. The third record-set, `sweep_traces`, holds **527 million** rows (one per digitised sample) — far too large to materialise in memory, so we read it *on demand* with predicate push-down when plotting individual sweeps. Every column in every table is linked back to a specific method step via the `method_description_object` entries in the data dictionary.

In [ ]:
# Load the dataset from the Croissant metadata file
# (uncomment the URL form if running remotely)
# ds = mlc.Dataset('https://sen.science/doi/10.71728/senscience.k7f3-9xqm/fair2.json')
ds = mlc.Dataset('./fair2.json')

# Resolve where the Parquet files physically live (package root, or the sibling wrangler output).
def resolve_data_dir():
    candidates = [
        'data/parquet',
        'resources/data/parquet',
        '../out_rt_recordings_20260703_133919/trn_whole_cell_patch_clamp_electrophysiology_juvenile_mouse_thalamic_reticular/data/parquet',
    ]
    for c in candidates:
        if Path(c).is_dir():
            return c
    hits = list(Path('.').rglob('sweep_traces.parquet'))
    return str(hits[0].parent) if hits else 'data/parquet'

DATA_DIR = resolve_data_dir()
print(f'Parquet directory: {DATA_DIR}')

# Load the two small metadata record-sets (skip the 527M-row sweep_traces).
dataframes = {}
for rs in ds.metadata.record_sets:
    name = rs.name.replace('.parquet', '')
    if name == 'sweep_traces':
        continue  # 527,232,374 rows — read on demand in the sweep viewer below
    try:
        df = pd.DataFrame(list(ds.records(rs.id)))
        df.columns = [col.split('/')[-1] for col in df.columns]
    except Exception:
        # Fallback: read the Parquet directly if the Croissant file references
        # aren't resolvable in this runtime.
        df = pd.read_parquet(f'{DATA_DIR}/{name}.parquet')
    dataframes[name] = df

mouse_metadata = dataframes['mouse_metadata']
sweep_metadata = dataframes['sweep_metadata']

# mlcroissant sometimes loads text columns as bytes — normalise here
for df in (mouse_metadata, sweep_metadata):
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

# On-demand reader for the raw traces: predicate push-down keeps this fast even at 527M rows.
def load_traces(cell, waves, cols=('wave_name', 'sample_index', 'time_s', 'value')):
    tbl = pq.read_table(
        f'{DATA_DIR}/sweep_traces.parquet',
        filters=[('igor_file_name', '=', cell), ('wave_name', 'in', list(waves))],
        columns=list(cols),
    )
    return tbl.to_pandas()

print(f"\nMetadata record sets loaded via mlcroissant / Parquet.")
print(f"  mouse_metadata : {len(mouse_metadata):>7,} rows × {len(mouse_metadata.columns)} cols")
print(f"  sweep_metadata : {len(sweep_metadata):>7,} rows × {len(sweep_metadata.columns)} cols")
print(f"  sweep_traces   : 527,232,374 rows × 5 cols  (read on demand)")

### 1.1 Data dictionary with method-step provenance
Every column in every table is linked to a specific method step via the schema.json's `method_description_object`. Here we surface the linkage in tabular form.

In [ ]:
# Load the schema.json to expose the data dictionary + method step links
schema = json.loads(Path('./schema.json').read_text())

# Method-step catalogue
step_index = {}
for section in schema.get('methods_description', []):
    for step in section.get('steps', []):
        step_index[step['id']] = {'section': section.get('name', ''), 'step': step.get('name', '')}
print(f'{len(step_index)} method steps declared in schema.methods_description')

# Build a wide table: (table, column, dataType, unit, → method step)
rows = []
for row in schema['data_dictionary']['data_dictionary_rows']:
    tname = row['file_object']['name'].replace('.parquet', '')
    for f in row['fields']:
        md_obj = f.get('method_description_object', {}) or {}
        rows.append({
            'table':       tname,
            'column':      f['variable_name'],
            'dataType':    f.get('dataType', '').split('/')[-1],
            'unit':        f.get('unit', {}).get('symbol', ''),
            'method_step': md_obj.get('name', '—'),
        })
dd_df = pd.DataFrame(rows)
display(Markdown(f'### {len(dd_df)} variables across {dd_df.table.nunique()} tables — each linked to a method step'))
display(dd_df)

### 1.2 Dataset Metadata
Inspect the dataset's top-level metadata to confirm name, DOI, and licence.

In [ ]:
metadata = ds.metadata.to_json()

name        = metadata.get('name', 'N/A')
description = metadata.get('description', 'N/A')
if isinstance(description, list):
    description = description[0].get('@value', description[0]) if description else 'N/A'

print(f"Name       : {name}")
print(f"Version    : {metadata.get('version', 'N/A')}")
print(f"License    : {metadata.get('license', 'N/A')}")
print(f"URL        : {metadata.get('url', 'N/A')}")
print(f"Creators   : {len(ds.metadata.creators)}")
for c in ds.metadata.creators:
    print(f"    - {c.name}")
print(f"\nDescription: {description}")

## 2. Data Preparation and Integration
We assemble a per-recording integrated summary by joining the sweep-level table onto the session-level table on the canonical Igor Pro file identifier, and derive per-recording summary statistics (sweeps per recording, distinct protocols, mean sampling rate).

In [ ]:
# --- Per-recording sweep summary --- #
# sweep_metadata has one row per recorded sweep; summarise to one row per recording.
sweep_metadata['sampling_rate_hz_num'] = pd.to_numeric(sweep_metadata['sampling_rate_hz'], errors='coerce')

rec_sweep_summary = (
    sweep_metadata.groupby('igor_file_name')
                  .agg(n_sweeps=('wave_name', 'size'),
                       n_protocols=('stimulus_name', 'nunique'),
                       mean_rate_hz=('sampling_rate_hz_num', 'mean'))
                  .reset_index()
)

# Merge onto the per-session animal metadata
recording_summary = mouse_metadata.merge(rec_sweep_summary, on='igor_file_name', how='left')

# Numeric coercions for the demographic fields
recording_summary['body_weight_g_num'] = pd.to_numeric(recording_summary['body_weight_g'], errors='coerce')

display(Markdown('### Recording summary (per neuron/session)'))
display(recording_summary[['igor_file_name', 'animal_id', 'sex', 'age_days', 'body_weight_g',
                           'neuron_recorded', 'initial_vm_mv', 'n_sweeps', 'n_protocols', 'mean_rate_hz']].head())

## 3. Exploratory Data Analysis (EDA)

### 3.1 Overview of the Integrated Dataset

In [ ]:
print("Recording summary:")
print(f"  {recording_summary.shape[0]} recordings × {recording_summary.shape[1]} fields")
print(f"  sex distribution     : {dict(recording_summary['sex'].value_counts())}")
print(f"  strain / genotype    : {dict(recording_summary['strain'].value_counts())} / {dict(recording_summary['genotype'].value_counts())}")
print(f"  recorded region      : {dict(recording_summary['neuron_recorded'].value_counts())}")
print(f"  age range (days)     : {int(recording_summary['age_days'].min())}–{int(recording_summary['age_days'].max())} (median {recording_summary['age_days'].median():.1f})")
print()
print("Sweep-level:")
print(f"  {len(sweep_metadata):,} sweeps across {sweep_metadata['igor_file_name'].nunique()} recordings")
print(f"  distinct stimulus protocols : {sweep_metadata['stimulus_name'].nunique()}")
print(f"  total digitised samples     : {int(sweep_metadata['n_samples'].sum()):,}  (= rows in sweep_traces)")

### 3.2 Data Visualization

#### 3.2.1 Cohort composition
Demographic and physiological range of the 22 recorded animals: postnatal age, body weight, and the resting membrane potential recorded at session start.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(recording_summary['age_days'].dropna(), bins=range(23, 36),
             color='#4C72B0', edgecolor='white', align='left')
axes[0].set_xlabel('Postnatal age (days)'); axes[0].set_ylabel('# recordings')
axes[0].set_title('Age distribution')

axes[1].hist(recording_summary['body_weight_g_num'].dropna(), bins=8,
             color='#DD8452', edgecolor='white')
axes[1].set_xlabel('Body weight (g)'); axes[1].set_ylabel('# recordings')
axes[1].set_title('Body weight')

axes[2].hist(recording_summary['initial_vm_mv'].dropna(), bins=8,
             color='#55A868', edgecolor='white')
axes[2].set_xlabel('Initial $V_m$ (mV)'); axes[2].set_ylabel('# recordings')
axes[2].set_title('Resting membrane potential')

plt.tight_layout(); plt.show()

#### 3.2.2 The standardised stimulus battery
Sweeps are distributed across 54 distinct stimulus protocols. Here we show the most-represented protocols by total sweep count.

In [ ]:
proto = (sweep_metadata.groupby('stimulus_name')
                        .agg(n_sweeps=('wave_name', 'size'),
                             n_recordings=('igor_file_name', 'nunique'))
                        .sort_values('n_sweeps', ascending=False))

top = proto.head(20)[::-1]
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top.index, top['n_sweeps'], color='#8172B3', edgecolor='white')
for y, (n, r) in enumerate(zip(top['n_sweeps'], top['n_recordings'])):
    ax.text(n + 40, y, f'{n:,} ({r} rec)', va='center', fontsize=7)
ax.set_xlabel('Sweeps'); ax.set_xlim(0, top['n_sweeps'].max() * 1.18)
ax.set_title(f'Stimulus battery — 20 most-represented protocols (of {sweep_metadata["stimulus_name"].nunique()})')
plt.tight_layout(); plt.show()

#### 3.2.3 Electrophysiology sweep viewer
Multi-protocol view of the current-clamp responses for a representative cell. Each panel shows one stimulus protocol; sweeps are colour-graded from earliest (dark) to latest (light). Traces are read on demand from the `sweep_traces` record-set (voltage channel), filtered to this cell and protocol via predicate push-down.

You can change `EXAMPLE_CELL` below to any `igor_file_name` that appears in `mouse_metadata`.

In [ ]:
EXAMPLE_CELL   = 'MPG210604Th_A_1'   # representative recording — used in Figure 3 of the data article
VOLTAGE_CHANNEL = 6                   # ch6 = recorded membrane potential (ch7 = injected current command)
protocols       = ['IV', 'IDRest', 'APWaveform', 'Delta']

if EXAMPLE_CELL not in set(sweep_metadata['igor_file_name']):
    EXAMPLE_CELL = sweep_metadata['igor_file_name'].iloc[0]
print(f'Rendering sweeps for cell {EXAMPLE_CELL}')

fig, axes = plt.subplots(len(protocols), 1, figsize=(9, 10))
cmap = plt.get_cmap('viridis')

for ax, proto in zip(axes, protocols):
    waves = (sweep_metadata[(sweep_metadata.igor_file_name == EXAMPLE_CELL) &
                            (sweep_metadata.stimulus_name == proto) &
                            (sweep_metadata.channel == VOLTAGE_CHANNEL)]
             .sort_values('wave_name')['wave_name'].unique()[:8])
    if len(waves):
        tr = load_traces(EXAMPLE_CELL, waves)
        for i, w in enumerate(waves):
            s = tr[tr.wave_name == w].sort_values('sample_index')
            ax.plot(s['time_s'], s['value'] * 1000.0, lw=0.5, color=cmap(i / max(len(waves) - 1, 1)))
    ax.set_title(f'{proto}  ({len(waves)} sweep{"s" if len(waves) != 1 else ""})', loc='left', fontsize=9)
    ax.set_ylabel('$V_m$ (mV)')
axes[-1].set_xlabel('Time (s)')
fig.suptitle(f'Representative whole-cell current-clamp traces · {EXAMPLE_CELL}', y=1.0)
plt.tight_layout(); plt.show()

#### 3.2.4 Recording composition
How the recordings themselves are composed: the number of sweeps per recording, and the acquisition sampling-rate split across all sweeps.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

spr = sweep_metadata.groupby('igor_file_name').size()
axes[0].hist(spr.values, bins=10, color='#4C72B0', edgecolor='white')
axes[0].set_xlabel('Sweeps per recording'); axes[0].set_ylabel('# recordings')
axes[0].set_title(f'Sweeps per recording (median {int(spr.median())})')

rate_bucket = np.where(sweep_metadata['sampling_rate_hz_num'] > 40000, '~50 kHz', '10 kHz')
rc = pd.Series(rate_bucket).value_counts()
axes[1].bar(rc.index, rc.values, color=['#DD8452', '#55A868'], width=0.6, edgecolor='white')
for x, v in enumerate(rc.values):
    axes[1].text(x, v + 200, f'{v:,}', ha='center', fontsize=9)
axes[1].set_ylabel('Sweeps'); axes[1].set_ylim(0, rc.values.max() * 1.15)
axes[1].set_title('Acquisition sampling rate')

plt.tight_layout(); plt.show()

## 4. Conclusion
This notebook walked through a first exploration of the LNMC MPG TRN whole-cell patch-clamp dataset:

- **Loading**: Croissant metadata (`fair2.json`) drove the ingestion of the two tidy metadata record-sets — 22 per-session `mouse_metadata` rows and 22,918 per-sweep `sweep_metadata` rows — while the 527-million-row `sweep_traces` table was read on demand with predicate push-down.
- **Data dictionary with method provenance**: every column in every table is linked to a specific step in `methods_description` via `method_description_object`.
- **Cohort composition**: the demographic and physiological range of the 22 male juvenile C57Bl/6J mice (postnatal day 23–34, resting $V_m$ −83 to −55 mV).
- **Stimulus battery**: the distribution of sweeps across the 54 distinct current-clamp protocols.
- **Electrophysiology viewer**: a multi-panel current-clamp trace display grouped by stimulus protocol, reading the raw voltage traces directly from the `sweep_traces` record-set.
- **Recording composition**: sweeps per recording and the 10 kHz / ~50 kHz acquisition-rate split.

Everything shown here is reproducible from the FAIR² package alone: the Croissant `recordSet` entries define the tabular schema (with method-step linkage), and the raw signal for every sweep is preserved in `sweep_traces` (and in the canonical NWB / Igor Pro assets). To extend, consider (a) extracting quantitative e-features (input resistance, rebound-burst latency, firing rate, adaptation) directly from the traces, (b) automated e-type classification driven by the protocol grouping, or (c) fitting and validating biophysical models of TRN firing against the standardised stimulus battery.